# 01. Первичный обзор и подготовка данных

В проекте используются два независимых кейса:

1. **Event-датасет интернет-магазина** — для продуктовых метрик, воронки, retention и сегментации.
2. **A/B-тест посадочной страницы** — для статистической проверки изменения конверсии.

На этом этапе мы:

- загрузим данные;
- проверим структуру и типы столбцов;
- найдём пропуски и дубликаты;
- подготовим даты;
- очистим A/B-тест;
- объединим A/B-данные со странами;
- сохраним очищенные таблицы.

> Два кейса нельзя объединять по `user_id`: идентификаторы относятся к разным исходным датасетам.

## Ожидаемая структура проекта

```text
product-analytics-funnel-retention-ab-testing/
├── data/
│   ├── raw/
│   │   ├── user_events_ecommerce_product_analytics.csv
│   │   ├── ab_data.csv
│   │   └── countries.csv
│   └── processed/
└── notebooks/
    └── 01_data_overview.ipynb
```

Запускай ноутбук из папки `notebooks` или из корня проекта — код автоматически определит путь.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

In [3]:
# Определяем корень проекта независимо от места запуска ноутбука.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

EVENTS_PATH = RAW_DIR / "user_events_ecommerce_product_analytics.csv"
AB_PATH = RAW_DIR / "ab_data.csv"
COUNTRIES_PATH = RAW_DIR / "countries.csv"

required_files = [EVENTS_PATH, AB_PATH, COUNTRIES_PATH]
missing_files = [str(path) for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "Не найдены файлы:\n" + "\n".join(missing_files)
    )

print("Корень проекта:", PROJECT_ROOT.resolve())
print("Все исходные файлы найдены.")

Корень проекта: E:\Projects SQL\product-analytics-funnel-retention-ab-testing
Все исходные файлы найдены.


## 1. Загрузка данных

In [6]:
events = pd.read_csv(EVENTS_PATH)
ab_data = pd.read_csv(AB_PATH)
countries = pd.read_csv(COUNTRIES_PATH)

print(f"events:    {events.shape[0]:,} строк × {events.shape[1]} столбцов")
print(f"ab_data:   {ab_data.shape[0]:,} строк × {ab_data.shape[1]} столбцов")
print(f"countries: {countries.shape[0]:,} строк × {countries.shape[1]} столбцов")

events:    320,000 строк × 12 столбцов
ab_data:   294,480 строк × 5 столбцов
countries: 290,586 строк × 2 столбцов


In [7]:
events.head()

,user_id,session_id,event_date,event_time,event_type,product_id,category,price,device,traffic_source,country,city
0,U110476,S2867825,2024-01-13,09:23:00,view,P3286,Men Pants,NaN,Mobile,Referral,USA,San Francisco
1,U100520,S1499914,2024-02-17,07:27:00,purchase,P4257,Casual Wear,"2,502.9400",Web,Email,USA,New York
2,U100106,S3678638,2024-12-23,14:25:00,add_to_cart,P3547,Women Pants,"4,815.0200",Web,Organic,UK,Manchester
3,U101519,S7374122,2024-02-19,12:15:00,view,P1711,Formal Wear,NaN,Mobile,Referral,UK,Manchester
4,U101291,S5918715,2024-11-17,21:06:00,visit,P2139,Men Pants,NaN,Mobile,Email,UK,London


In [8]:
ab_data.head()

,user_id,timestamp,group,landing_page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,661590,55:06.2,treatment,new_page,0
3,853541,28:03.1,treatment,new_page,0
4,864975,52:26.2,control,old_page,1


In [9]:
countries.head()

,user_id,country
0,834778,UK
1,928468,US
2,822059,UK
3,711597,UK
4,710616,UK


## 2. Универсальная функция первичного обзора

In [10]:
def dataframe_overview(df: pd.DataFrame) -> pd.DataFrame:
    """Возвращает краткую информацию о столбцах таблицы."""
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100).round(2),
        "n_unique": df.nunique(dropna=True),
    }).sort_values(["missing_pct", "n_unique"], ascending=[False, False])

### Event-датасет

In [11]:
dataframe_overview(events)

,dtype,non_null,missing,missing_pct,n_unique
price,float64,121820,198180,61.9300,106160
session_id,object,320000,0,0.0000,314284
user_id,object,320000,0,0.0000,12000
product_id,object,320000,0,0.0000,9000
event_time,object,320000,0,0.0000,1440
event_date,object,320000,0,0.0000,366
city,object,320000,0,0.0000,15
event_type,object,320000,0,0.0000,5
category,object,320000,0,0.0000,5
country,object,320000,0,0.0000,5


### A/B-тест

In [12]:
dataframe_overview(ab_data)

,dtype,non_null,missing,missing_pct,n_unique
user_id,int64,294480,0,0.0000,290585
timestamp,object,294480,0,0.0000,35993
group,object,294480,0,0.0000,2
landing_page,object,294480,0,0.0000,2
converted,int64,294480,0,0.0000,2


### Страны

In [13]:
dataframe_overview(countries)

,dtype,non_null,missing,missing_pct,n_unique
user_id,int64,290586,0,0.0000,290585
country,object,290586,0,0.0000,3


## 3. Проверка event-датасета

Пропуски в `price` не удаляем автоматически. В этом датасете цена заполнена у событий:

- `add_to_cart`;
- `checkout`;
- `purchase`.

У `visit` и `view` цена отсутствует по логике формирования данных.

In [14]:
event_counts = (
    events["event_type"]
    .value_counts()
    .rename_axis("event_type")
    .reset_index(name="events")
)

event_counts["share_pct"] = (
    event_counts["events"] / len(events) * 100
).round(2)

event_counts

,event_type,events,share_pct
0,view,102371,31.9900
1,visit,95809,29.9400
2,add_to_cart,57809,18.0700
3,checkout,38501,12.0300
4,purchase,25510,7.9700


In [15]:
price_quality = (
    events.groupby("event_type", observed=True)
    .agg(
        rows=("event_type", "size"),
        price_filled=("price", "count"),
        price_missing=("price", lambda series: series.isna().sum()),
    )
)

price_quality["price_missing_pct"] = (
    price_quality["price_missing"] / price_quality["rows"] * 100
).round(2)

price_quality

,rows,price_filled,price_missing,price_missing_pct
event_type,,,,
add_to_cart,57809,57809,0,0.0000
checkout,38501,38501,0,0.0000
purchase,25510,25510,0,0.0000
view,102371,0,102371,100.0000
visit,95809,0,95809,100.0000


In [16]:
# Объединяем дату и время в один временной столбец.
events["event_datetime"] = pd.to_datetime(
    events["event_date"].astype(str) + " " + events["event_time"].astype(str),
    errors="coerce",
)

print("Некорректных дат:", events["event_datetime"].isna().sum())
print("Начало периода:", events["event_datetime"].min())
print("Конец периода:", events["event_datetime"].max())

Некорректных дат: 0
Начало периода: 2024-01-01 00:00:00
Конец периода: 2024-12-31 23:57:00


In [17]:
event_exact_duplicates = events.duplicated().sum()

event_key_duplicates = events.duplicated(
    subset=[
        "user_id",
        "session_id",
        "event_datetime",
        "event_type",
        "product_id",
    ]
).sum()

print("Полных дубликатов:", event_exact_duplicates)
print("Дубликатов по ключевым полям события:", event_key_duplicates)
print("Уникальных пользователей:", events["user_id"].nunique())
print("Уникальных сессий:", events["session_id"].nunique())
print("Уникальных товаров:", events["product_id"].nunique())

Полных дубликатов: 0
Дубликатов по ключевым полям события: 0
Уникальных пользователей: 12000
Уникальных сессий: 314284
Уникальных товаров: 9000


In [18]:
events_per_session = events.groupby("session_id").size()

events_per_session.describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99])

count   314,284.0000
mean          1.0182
std           0.1354
min           1.0000
50%           1.0000
75%           1.0000
90%           1.0000
95%           1.0000
99%           2.0000
max           3.0000
dtype: float64

### Вывод по `session_id`

Большинство идентификаторов сессий содержат одно событие. Поэтому стандартная сессионная воронка будет малоинформативной.

В следующих ноутбуках основную воронку будем строить:

1. на уровне пользователя;
2. с обязательным соблюдением порядка событий;
3. при необходимости — в выбранном временном окне.

In [19]:
# Создаём несколько удобных временных признаков.
events["event_date"] = events["event_datetime"].dt.normalize()
events["event_month"] = events["event_datetime"].dt.to_period("M").astype(str)
events["event_week"] = events["event_datetime"].dt.to_period("W").astype(str)
events["event_hour"] = events["event_datetime"].dt.hour
events["day_of_week"] = events["event_datetime"].dt.day_name()

# Сортировка важна для последующего расчёта последовательной воронки.
events_clean = events.sort_values(
    ["user_id", "event_datetime", "event_type"]
).reset_index(drop=True)

events_clean.head()

,user_id,session_id,event_date,event_time,event_type,product_id,category,price,device,traffic_source,country,city,event_datetime,event_month,event_week,event_hour,day_of_week
0,U100000,S2139541,2024-01-24,05:14:00,visit,P6605,Jeans,NaN,Mobile,Ads,Australia,Melbourne,2024-01-24 05:14:00,2024-01,2024-01-22/2024-01-28,5,Wednesday
1,U100000,S1303026,2024-01-26,02:09:00,visit,P6606,Formal Wear,NaN,Web,Organic,Germany,Berlin,2024-01-26 02:09:00,2024-01,2024-01-22/2024-01-28,2,Friday
2,U100000,S6081409,2024-01-27,21:46:00,view,P4046,Women Pants,NaN,Mobile,Ads,UK,Manchester,2024-01-27 21:46:00,2024-01,2024-01-22/2024-01-28,21,Saturday
3,U100000,S5883935,2024-02-06,07:56:00,purchase,P3594,Formal Wear,872.8100,Mobile,Ads,Australia,Melbourne,2024-02-06 07:56:00,2024-02,2024-02-05/2024-02-11,7,Tuesday
4,U100000,S6778838,2024-02-23,19:23:00,view,P6422,Formal Wear,NaN,Web,Ads,USA,Austin,2024-02-23 19:23:00,2024-02,2024-02-19/2024-02-25,19,Friday


## 4. Проверка и очистка A/B-теста

In [20]:
print("Распределение по группам:")
display(ab_data["group"].value_counts())

print("\nРаспределение по страницам:")
display(ab_data["landing_page"].value_counts())

print("\nИсходная конверсия по группам:")
display(
    ab_data.groupby("group")
    .agg(
        rows=("user_id", "size"),
        users=("user_id", "nunique"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean"),
    )
)

Распределение по группам:


group
treatment    147278
control      147202
Name: count, dtype: int64


Распределение по страницам:


landing_page
new_page    147241
old_page    147239
Name: count, dtype: int64


Исходная конверсия по группам:


,rows,users,conversions,conversion_rate
group,,,,
control,147202,146195,17723,0.1204
treatment,147278,146285,17514,0.1189


По дизайну эксперимента допустимы только две комбинации:

- `control` + `old_page`;
- `treatment` + `new_page`.

Строки с другими сочетаниями нарушают логику эксперимента и будут удалены.

In [21]:
valid_assignment = (
    ((ab_data["group"] == "control") & (ab_data["landing_page"] == "old_page"))
    | ((ab_data["group"] == "treatment") & (ab_data["landing_page"] == "new_page"))
)

print("Корректных строк:", valid_assignment.sum())
print("Некорректных сочетаний group/page:", (~valid_assignment).sum())

Корректных строк: 290587
Некорректных сочетаний group/page: 3893


In [22]:
ab_valid = ab_data.loc[valid_assignment].copy()

duplicate_user_rows = ab_valid["user_id"].duplicated(keep=False)

print("Строк с повторяющимися user_id после удаления несоответствий:",
      duplicate_user_rows.sum())
print("Уникальных повторяющихся user_id:",
      ab_valid.loc[duplicate_user_rows, "user_id"].nunique())

ab_valid.loc[duplicate_user_rows].sort_values("user_id")

Строк с повторяющимися user_id после удаления несоответствий: 4
Уникальных повторяющихся user_id: 2


,user_id,timestamp,group,landing_page,converted
294478,759899,20:29.0,treatment,new_page,0
250001,759899,07:36.1,treatment,new_page,0
2893,773192,55:59.6,treatment,new_page,0
1899,773192,37:58.8,treatment,new_page,0


Поскольку полноценная календарная дата в `timestamp` потеряна, определить более раннюю запись надёжно нельзя. Поэтому для каждого повторяющегося пользователя оставляем первую строку и явно фиксируем это решение.

In [23]:
ab_clean = (
    ab_valid
    .drop_duplicates(subset="user_id", keep="first")
    .reset_index(drop=True)
)

print("Строк после очистки:", len(ab_clean))
print("Уникальных пользователей:", ab_clean["user_id"].nunique())

Строк после очистки: 290585
Уникальных пользователей: 290585


## 5. Подготовка таблицы стран и объединение

In [24]:
print("Дубликатов user_id в countries:", countries["user_id"].duplicated().sum())

countries_clean = (
    countries
    .drop_duplicates(subset="user_id", keep="first")
    .reset_index(drop=True)
)

ab_clean = ab_clean.merge(
    countries_clean,
    on="user_id",
    how="left",
    validate="one_to_one",
)

print("Размер объединённой таблицы:", ab_clean.shape)
print("Пользователей без страны:", ab_clean["country"].isna().sum())

Дубликатов user_id в countries: 1
Размер объединённой таблицы: (290585, 6)
Пользователей без страны: 0


In [25]:
ab_summary = (
    ab_clean.groupby("group", observed=True)
    .agg(
        users=("user_id", "nunique"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean"),
    )
)

ab_summary["conversion_rate_pct"] = (
    ab_summary["conversion_rate"] * 100
).round(3)

ab_summary

,users,conversions,conversion_rate,conversion_rate_pct
group,,,,
control,145274,17489,0.1204,12.0390
treatment,145311,17264,0.1188,11.8810


In [26]:
country_summary = (
    ab_clean.groupby(["group", "country"], observed=True)
    .agg(
        users=("user_id", "nunique"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean"),
    )
    .reset_index()
)

country_summary["conversion_rate_pct"] = (
    country_summary["conversion_rate"] * 100
).round(3)

country_summary

,group,country,users,conversions,conversion_rate,conversion_rate_pct
0,control,CA,7198,855,0.1188,11.8780
1,control,UK,36360,4364,0.1200,12.0020
2,control,US,101716,12270,0.1206,12.0630
3,treatment,CA,7301,817,0.1119,11.1900
4,treatment,UK,36106,4375,0.1212,12.1170
5,treatment,US,101904,12072,0.1185,11.8460


## 6. Автоматические проверки качества

In [27]:
# Event-датасет
assert events_clean["event_datetime"].notna().all()
assert events_clean.duplicated().sum() == 0
assert set(events_clean["event_type"].unique()) == {
    "visit", "view", "add_to_cart", "checkout", "purchase"
}

# A/B-тест
assert ab_clean["user_id"].is_unique
assert ab_clean["country"].notna().all()
assert ab_clean["converted"].isin([0, 1]).all()

assert (
    (ab_clean["group"] == "control")
    == (ab_clean["landing_page"] == "old_page")
).all()

assert (
    (ab_clean["group"] == "treatment")
    == (ab_clean["landing_page"] == "new_page")
).all()

print("Все проверки качества пройдены.")

Все проверки качества пройдены.


## 7. Сохранение очищенных таблиц

In [28]:
EVENTS_CLEAN_PATH = PROCESSED_DIR / "events_clean.csv"
AB_CLEAN_PATH = PROCESSED_DIR / "ab_test_clean.csv"

events_clean.to_csv(EVENTS_CLEAN_PATH, index=False)
ab_clean.to_csv(AB_CLEAN_PATH, index=False)

print("Сохранено:")
print(EVENTS_CLEAN_PATH.resolve())
print(AB_CLEAN_PATH.resolve())

Сохранено:
E:\Projects SQL\product-analytics-funnel-retention-ab-testing\data\processed\events_clean.csv
E:\Projects SQL\product-analytics-funnel-retention-ab-testing\data\processed\ab_test_clean.csv


## Итоги этапа

### Event-датасет

- данные охватывают весь 2024 год;
- временные поля успешно преобразованы;
- полных дубликатов событий нет;
- пропуски цены объясняются типом события;
- большинство `session_id` содержит одно событие;
- для основной продуктовой воронки будем использовать уровень пользователя.

### A/B-тест

- удалены несовпадения между группой и страницей;
- устранены повторные пользователи;
- добавлены страны;
- после очистки осталось **290 585 уникальных пользователей**;
- контрольная группа имеет немного более высокую сырую конверсию, но статистический вывод пока не делаем.

Следующий ноутбук:

```text
02_product_metrics_and_funnel.ipynb
```

В нём рассчитаем базовые продуктовые метрики и построим последовательную воронку:

`visit → view → add_to_cart → checkout → purchase`.
